In [118]:
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer(as_frame=True)
x = data.data
y = data.target

In [119]:
import pandas as pd

print("=== Dataset preview (Breast Cancer) ===")
print(f"X shape: {x.shape}")
print(f"y shape: {getattr(y, 'shape', None)}")

# Tabela com as primeiras linhas
display(x.head(8))

# Estatisticas descritivas
display(x.describe().T)

# Distribuicao do alvo
y_series = y if isinstance(y, pd.Series) else pd.Series(y)
display(y_series.value_counts(dropna=False).rename_axis("classe").to_frame("contagem"))

# Valores ausentes (top 10)
missing = x.isna().sum().sort_values(ascending=False)
display(missing.head(10).rename("missing"))
print(f"y missing: {y_series.isna().sum()}")

=== Dataset preview (Breast Cancer) ===
X shape: (569, 30)
y shape: (569,)


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678
5,12.45,15.70,82.57,477.1,0.12780,0.17000,0.15780,0.08089,0.2087,0.07613,...,15.47,23.75,103.40,741.6,0.1791,0.5249,0.5355,0.1741,0.3985,0.12440
6,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.11270,0.07400,0.1794,0.05742,...,22.88,27.66,153.20,1606.0,0.1442,0.2576,0.3784,0.1932,0.3063,0.08368
7,13.71,20.83,90.20,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,0.07451,...,17.06,28.14,110.60,897.0,0.1654,0.3682,0.2678,0.1556,0.3196,0.11510


,count,mean,std,min,25%,50%,75%,max
mean radius,569.0,14.127292,3.524049,6.981000,11.700000,13.370000,15.780000,28.11000
mean texture,569.0,19.289649,4.301036,9.710000,16.170000,18.840000,21.800000,39.28000
mean perimeter,569.0,91.969033,24.298981,43.790000,75.170000,86.240000,104.100000,188.50000
mean area,569.0,654.889104,351.914129,143.500000,420.300000,551.100000,782.700000,2501.00000
mean smoothness,569.0,0.096360,0.014064,0.052630,0.086370,0.095870,0.105300,0.16340
mean compactness,569.0,0.104341,0.052813,0.019380,0.064920,0.092630,0.130400,0.34540
mean concavity,569.0,0.088799,0.079720,0.000000,0.029560,0.061540,0.130700,0.42680
mean concave points,569.0,0.048919,0.038803,0.000000,0.020310,0.033500,0.074000,0.20120
mean symmetry,569.0,0.181162,0.027414,0.106000,0.161900,0.179200,0.195700,0.30400
mean fractal dimension,569.0,0.062798,0.007060,0.049960,0.057700,0.061540,0.066120,0.09744


,contagem
classe,
1,357
0,212


mean radius               0
mean texture              0
mean perimeter            0
mean area                 0
mean smoothness           0
mean compactness          0
mean concavity            0
mean concave points       0
mean symmetry             0
mean fractal dimension    0
Name: missing, dtype: int64

y missing: 0


In [120]:
# Algoritmo do perceptron (funcoes)
import random

def kfold_split(X, y, k=5, seed=42):
    idx = list(range(len(X)))
    random.seed(seed)
    random.shuffle(idx)
    folds = [[] for _ in range(k)]
    for i, index in enumerate(idx):
        folds[i % k].append(index)
    return folds

# Normalizacao (z-score) baseada no treino
def fit_standardizer(X):
    n_rows = len(X)
    n_cols = len(X[0])
    means = [0.0] * n_cols
    stds = [0.0] * n_cols

    for row in X:
        for j in range(n_cols):
            means[j] += row[j]
    for j in range(n_cols):
        means[j] /= n_rows

    for row in X:
        for j in range(n_cols):
            diff = row[j] - means[j]
            stds[j] += diff * diff
    for j in range(n_cols):
        stds[j] = (stds[j] / n_rows) ** 0.5
        if stds[j] == 0:
            stds[j] = 1.0
    return means, stds

def transform_standardizer(X, means, stds):
    Xz = []
    for row in X:
        Xz.append([(row[j] - means[j]) / stds[j] for j in range(len(row))])
    return Xz

# Predicao com o perceptron treinado
def prever_perceptron(X, w, b):
    preds = []
    for xi in X:
        s = b
        for j in range(len(w)):
            s += w[j] * xi[j]
        y_pred = 1 if s >= 0 else 0  # funcao de ativacao (degrau)
        preds.append(y_pred)
    return preds

# Acuracia simples
def acuracia(y_true, y_pred):
    corretos = 0
    for yt, yp in zip(y_true, y_pred):
        if yt == yp:
            corretos += 1
    return corretos / len(y_true) if y_true else 0.0

# Treinamento do perceptron
def treinar_perceptron(X, y, lr=0.01, epochs=20):
    n_features = len(X[0])
    # Pesos (um peso por entrada)
    w = [0.0] * n_features
    # Bias (termo independente)
    b = 0.0

    for epoch in range(epochs):
        erros = 0
        for xi, yi in zip(X, y):
            # Soma ponderada: entradas * pesos + bias
            s = b
            for j in range(n_features):
                s += w[j] * xi[j]

            # Funcao de ativacao (degrau)
            y_pred = 1 if s >= 0 else 0

            # Erro (diferenca entre alvo e predicao)
            erro = yi - y_pred
            if erro != 0:
                erros += 1
                # Atualiza pesos e bias
                for j in range(n_features):
                    w[j] += lr * erro * xi[j]
                b += lr * erro
        print(f"Epoch {epoch+1:02d} | erros={erros}")
    return w, b

def executar_kfold(X, y, k, lr, epochs, seed=42):
    folds = kfold_split(X, y, k=k, seed=seed)
    accs = []

    for fold_idx in range(k):
        test_idx = set(folds[fold_idx])
        train_idx = [i for i in range(len(X)) if i not in test_idx]

        X_train = [X[i] for i in train_idx]
        y_train = [y[i] for i in train_idx]
        X_test = [X[i] for i in test_idx]
        y_test = [y[i] for i in test_idx]

        # Normaliza usando apenas o treino
        means, stds = fit_standardizer(X_train)
        X_train_z = transform_standardizer(X_train, means, stds)
        X_test_z = transform_standardizer(X_test, means, stds)

        print(f"\nFold {fold_idx+1}/{k}")
        w, b = treinar_perceptron(X_train_z, y_train, lr=lr, epochs=epochs)
        y_pred_test = prever_perceptron(X_test_z, w, b)
        acc = acuracia(y_test, y_pred_test)
        accs.append(acc)
        print(f"Acuracia (fold {fold_idx+1}): {acc:.4f}")

    media = sum(accs) / len(accs) if accs else 0.0
    print(f"\nAcuracia media ({k}-fold): {media:.4f}")
    return media

In [121]:
# Treinos e testes (k-fold)

# Entradas (features) vindas da base importada
X = x.values.tolist()
y = y.tolist()

# Teste rapido de hiperparametros
configs = [
    {"k": 5, "lr": 0.01, "epochs": 20},
    {"k": 5, "lr": 0.01, "epochs": 40},
    {"k": 5, "lr": 0.05, "epochs": 20},
    {"k": 5, "lr": 0.005, "epochs": 40},
]

for cfg in configs:
    print("\n============================")
    print(f"Config: k={cfg['k']} | lr={cfg['lr']} | epochs={cfg['epochs']}")
    executar_kfold(X, y, k=cfg["k"], lr=cfg["lr"], epochs=cfg["epochs"], seed=42)


Config: k=5 | lr=0.01 | epochs=20

Fold 1/5
Epoch 01 | erros=30
Epoch 02 | erros=17
Epoch 03 | erros=20
Epoch 04 | erros=14
Epoch 05 | erros=18
Epoch 06 | erros=15
Epoch 07 | erros=15
Epoch 08 | erros=12
Epoch 09 | erros=15
Epoch 10 | erros=12
Epoch 11 | erros=21
Epoch 12 | erros=15
Epoch 13 | erros=13
Epoch 14 | erros=15
Epoch 15 | erros=11
Epoch 16 | erros=16
Epoch 17 | erros=13
Epoch 18 | erros=14
Epoch 19 | erros=12
Epoch 20 | erros=15
Acuracia (fold 1): 0.9474

Fold 2/5
Epoch 01 | erros=23
Epoch 02 | erros=21
Epoch 03 | erros=13
Epoch 04 | erros=16
Epoch 05 | erros=15
Epoch 06 | erros=12
Epoch 07 | erros=21
Epoch 08 | erros=10
Epoch 09 | erros=12
Epoch 10 | erros=12
Epoch 11 | erros=12
Epoch 12 | erros=9
Epoch 13 | erros=15
Epoch 14 | erros=18
Epoch 15 | erros=16
Epoch 16 | erros=12
Epoch 17 | erros=15
Epoch 18 | erros=9
Epoch 19 | erros=10
Epoch 20 | erros=11
Acuracia (fold 2): 0.9825

Fold 3/5
Epoch 01 | erros=29
Epoch 02 | erros=16
Epoch 03 | erros=18
Epoch 04 | erros=16
Epoch

In [122]:
# Tabela completa: previsto vs real (modelo final)

# Garante que o modelo final exista
try:
    w_final
    b_final
    means_final
    stds_final
except NameError:
    lr_final = 0.01
    epochs_final = 20
    means_final, stds_final = fit_standardizer(X)
    X_z = transform_standardizer(X, means_final, stds_final)
    w_final, b_final = treinar_perceptron(X_z, y, lr=lr_final, epochs=epochs_final)

# Prediz todas as linhas
X_all_z = transform_standardizer(X, means_final, stds_final)
y_pred_all = prever_perceptron(X_all_z, w_final, b_final)

label_map = {i: name for i, name in enumerate(data.target_names)}
df_resultados = pd.DataFrame({
    "indice": list(range(len(X))),
    "real": [label_map[v] for v in y],
    "predito": [label_map[v] for v in y_pred_all],
})
df_resultados["acertou"] = df_resultados["real"] == df_resultados["predito"]

# Exporta os resultados
output_csv = "predicoes_resultados.csv"
df_resultados.to_csv(output_csv, index=False)
print(f"Resultados exportados em: {output_csv}")

df_resultados

Resultados exportados em: predicoes_resultados.csv


,indice,real,predito,acertou
0,0,malignant,malignant,True
1,1,malignant,malignant,True
2,2,malignant,malignant,True
3,3,malignant,malignant,True
4,4,malignant,malignant,True
...,...,...,...,...
564,564,malignant,malignant,True
565,565,malignant,malignant,True
566,566,malignant,malignant,True
567,567,malignant,malignant,True
